# Video-Level Aggregation & Voting Evaluation
### Evaluating EFFResNet-ViT with Video-Level Temporal Aggregation

**Methodology:**
1. Load the optimal trained baseline checkpoint (`best_effresnet_vit.pth`).
2. Run inference on all held-out validation face frames across videos.
3. Compute both **Frame-Level** metrics and **Video-Level** aggregated metrics (Mean Probability Voting & Majority Hard Voting).
4. Benchmark directly against the original IEEE SMART-2025 paper (Table II: 98.4%).

In [ ]:
import os, sys, time, json, random
from pathlib import Path
from typing import List, Tuple, Dict
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import EfficientNet_B0_Weights, ResNet50_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print('PyTorch Version :', torch.__version__)
print('Device          :', device)
if torch.cuda.is_available():
    print('GPU Device      :', torch.cuda.get_device_name(0))
    print('VRAM Available  :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
print('=' * 65)

In [ ]:
# ── CHECKPOINT DISCOVERY ─────────────────────────────────────────
def find_checkpoint() -> Path:
    kaggle_input = Path('/kaggle/input')
    for p in kaggle_input.rglob('best_effresnet_vit.pth'):
        if p.is_file():
            return p
    # Check working directory or local candidates
    for p in Path('.').rglob('best_effresnet_vit.pth'):
        if p.is_file():
            return p
    raise FileNotFoundError('Could not locate best_effresnet_vit.pth under /kaggle/input')

CHECKPOINT_PATH = find_checkpoint()
print('Located Trained Model Checkpoint at:', CHECKPOINT_PATH)
print(f'Checkpoint File Size: {CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB')

In [ ]:
# ── DATASET DISCOVERY & VALIDATION SPLIT (MATCHING TRAINING SEED) ──
def locate_dataset_root() -> Path:
    for p in [
        Path('/kaggle/input/faceforencispp-extracted-frames'),
        Path('/kaggle/input/datasets/adham7elmy/faceforencispp-extracted-frames'),
    ]:
        if p.exists() and (p / 'real').exists():
            return p
    for sub in Path('/kaggle/input').rglob('real'):
        if sub.is_dir() and (sub.parent / 'fake').exists():
            return sub.parent
    raise FileNotFoundError('FaceForensics++ dataset not found under /kaggle/input')

DATASET_ROOT = locate_dataset_root()
print('Dataset Root:', DATASET_ROOT)

def build_validation_manifest(
    root: Path,
    max_frames_per_video: int = 15,
    val_split: float = 0.2,
    seed: int = 42
):
    random.seed(seed)
    real_dir = root / 'real'
    fake_dir = root / 'fake'
    
    real_videos = {}
    if real_dir.exists():
        for vid in sorted(real_dir.iterdir()):
            if vid.is_dir():
                frames = sorted([str(p) for p in vid.glob('*.png')])
                if frames:
                    real_videos[vid.name] = frames
                    
    fake_videos = {}
    if fake_dir.exists():
        for method in sorted(fake_dir.iterdir()):
            if method.is_dir():
                for vid in sorted(method.iterdir()):
                    if vid.is_dir():
                        frames = sorted([str(p) for p in vid.glob('*.png')])
                        if frames:
                            fake_videos[f'{method.name}_{vid.name}'] = frames
                            
    real_keys = list(real_videos.keys())
    random.shuffle(real_keys)
    n_real_val = max(1, int(len(real_keys) * val_split))
    val_real_keys = set(real_keys[:n_real_val])
    train_real_keys = set(real_keys[n_real_val:])
    
    fake_keys = list(fake_videos.keys())
    random.shuffle(fake_keys)
    n_fake_val = max(1, int(len(fake_keys) * val_split))
    val_fake_keys = set(fake_keys[:n_fake_val])
    train_fake_keys = set(fake_keys[n_fake_val:])
    
    def sample_from_keys(keys, src, label):
        samples = []
        for k in keys:
            fl = src[k]
            if len(fl) <= max_frames_per_video:
                chosen = fl
            else:
                step = len(fl) / max_frames_per_video
                chosen = [fl[int(i * step)] for i in range(max_frames_per_video)]
            for f in chosen:
                samples.append((f, label, k))
        return samples
        
    tr_real = sample_from_keys(train_real_keys, real_videos, 0)
    va_real = sample_from_keys(val_real_keys, real_videos, 0)
    tr_fake = sample_from_keys(train_fake_keys, fake_videos, 1)
    va_fake = sample_from_keys(val_fake_keys, fake_videos, 1)
    
    n_tr = min(len(tr_real), len(tr_fake))
    train_samples = random.sample(tr_real, n_tr) + random.sample(tr_fake, n_tr)
    
    n_va = min(len(va_real), len(va_fake))
    val_samples = random.sample(va_real, n_va) + random.sample(va_fake, n_va)
    
    random.shuffle(train_samples)
    random.shuffle(val_samples)
    
    return val_samples

val_entries = build_validation_manifest(DATASET_ROOT, max_frames_per_video=15)
unique_vids = len(set(e[2] for e in val_entries))
print(f'Total Held-Out Validation Frames : {len(val_entries):,}')
print(f'Total Unique Validation Videos   : {unique_vids:,}')

In [ ]:
# ── DATASET & DATALOADER ─────────────────────────────────────────
class FFPPVideoEvalDataset(Dataset):
    def __init__(self, entries, transform=None):
        self.entries = entries
        self.transform = transform
    def __len__(self): return len(self.entries)
    def __getitem__(self, idx):
        path, label, vid_id = self.entries[idx]
        try:    img = Image.open(path).convert('RGB')
        except: img = Image.new('RGB', (224, 224))
        if self.transform: img = self.transform(img)
        else: img = transforms.ToTensor()(img)
        return img, torch.tensor(label, dtype=torch.float32), vid_id

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_loader = DataLoader(
    FFPPVideoEvalDataset(val_entries, val_tf),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)
print(f'Validation Batches: {len(val_loader)}')

In [ ]:
# ── MODEL ARCHITECTURE DEFINITION & CHECKPOINT LOADING ───────────
class ConvLocalityBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = nn.GELU()
        self.depthwise = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.GELU()
    def forward(self, x): return self.act2(self.bn2(self.depthwise(self.act1(self.bn1(self.conv1(x))))))

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        normed = self.norm1(x)
        attn_out, _ = self.mha(normed, normed, normed)
        x = x + self.dropout1(attn_out)
        x = x + self.mlp(self.norm2(x))
        return x

class EFFResNetViT(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, num_vit_layers=4, mlp_ratio=4.0, dropout=0.1, pretrained=False):
        super().__init__()
        self.embed_dim = embed_dim
        eff = models.efficientnet_b0(weights=None)
        self.eff_features = eff.features
        res = models.resnet50(weights=None)
        self.res_stem = nn.Sequential(
            res.conv1, res.bn1, res.relu, res.maxpool,
            res.layer1, res.layer2, res.layer3, res.layer4
        )
        self.fusion_neck = ConvLocalityBlock(1280 + 2048, embed_dim)
        self.num_patches = 7 * 7
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.transformer_layers = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_vit_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.post_attn_locality = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, groups=embed_dim, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.GELU()
        )
        self.feat_proj = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        B = x.size(0)
        f_eff = self.eff_features(x)
        f_res = self.res_stem(x)
        if f_eff.shape[2:] != f_res.shape[2:]:
            f_eff = F.interpolate(f_eff, size=f_res.shape[2:], mode='bilinear', align_corners=False)
        f_fused = self.fusion_neck(torch.cat([f_eff, f_res], dim=1))
        H, W = f_fused.shape[2], f_fused.shape[3]
        tokens = f_fused.flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)
        tokens = self.pos_dropout(tokens + self.pos_embed[:, :tokens.size(1), :])
        for layer in self.transformer_layers: tokens = layer(tokens)
        tokens = self.norm(tokens)
        cls_out = tokens[:, 0]
        spatial_tokens = tokens[:, 1:]
        spatial_grid = spatial_tokens.transpose(1, 2).reshape(B, self.embed_dim, H, W)
        spatial_recovered = self.post_attn_locality(spatial_grid)
        spatial_pool = F.adaptive_avg_pool2d(spatial_recovered, (1, 1)).flatten(1)
        feat = self.feat_proj(torch.cat([cls_out, spatial_pool], dim=1))
        return self.classifier(feat)

model = EFFResNetViT(embed_dim=256, num_heads=8, num_vit_layers=4, pretrained=False).to(device)
try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Successfully loaded model weights from Epoch {checkpoint.get("epoch", "unknown")}')
else:
    model.load_state_dict(checkpoint)
    print('Successfully loaded model state_dict directly.')
model.eval()

In [ ]:
# ── INFERENCE EXECUTION & DUAL PREDICTION RECORDING ───────────────
all_probs = []
all_labels = []
all_vids = []

t0 = time.time()
print('Running inference on validation set...')
with torch.no_grad():
    for step, (imgs, lbls, vids) in enumerate(val_loader):
        imgs = imgs.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(imgs)
            probs = torch.sigmoid(logits)
        all_probs.extend(probs.squeeze(1).cpu().numpy().tolist())
        all_labels.extend(lbls.numpy().tolist())
        all_vids.extend(vids)
        if (step + 1) % 50 == 0:
            print(f'Processed [{step+1}/{len(val_loader)}] batches...')

elapsed = time.time() - t0
print(f'Inference finished in {elapsed:.1f}s ({len(all_probs)/elapsed:.1f} frames/sec)')

In [ ]:
# ── METRICS CALCULATION: FRAME-LEVEL VS VIDEO-LEVEL AGGREGATION ──
def compute_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred) * 100.0
    prec = precision_score(y_true, y_pred, zero_division=0) * 100.0
    rec = recall_score(y_true, y_pred, zero_division=0) * 100.0
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100.0
    auc = roc_auc_score(y_true, y_prob) * 100.0 if len(set(y_true)) > 1 else 50.0
    cm = confusion_matrix(y_true, y_pred)
    return {
        'accuracy': round(acc, 2),
        'precision': round(prec, 2),
        'recall': round(rec, 2),
        'f1_score': round(f1, 2),
        'auc_roc': round(auc, 2),
        'confusion_matrix': cm.tolist()
    }

# 1. Frame-Level Metrics
frame_metrics = compute_metrics(np.array(all_labels), np.array(all_probs))

# 2. Group predictions by Video ID
video_probs_dict = defaultdict(list)
video_labels_dict = {}
for p, y, vid in zip(all_probs, all_labels, all_vids):
    video_probs_dict[vid].append(p)
    video_labels_dict[vid] = y

sorted_vids = sorted(video_probs_dict.keys())
v_true = np.array([video_labels_dict[v] for v in sorted_vids])

# Mean Probability Voting
v_mean_prob = np.array([np.mean(video_probs_dict[v]) for v in sorted_vids])
video_mean_metrics = compute_metrics(v_true, v_mean_prob)

# Median Probability Voting
v_median_prob = np.array([np.median(video_probs_dict[v]) for v in sorted_vids])
video_median_metrics = compute_metrics(v_true, v_median_prob)

# Majority Hard Voting (count frames >= 0.5)
v_maj_pred = np.array([int(np.mean([p >= 0.5 for p in video_probs_dict[v]]) >= 0.5) for v in sorted_vids])
video_maj_acc = round(accuracy_score(v_true, v_maj_pred) * 100.0, 2)

print('=' * 80)
print('EVALUATION RESULTS:')
print(f'1. Single-Frame Evaluation  : Accuracy = {frame_metrics["accuracy"]}% | AUC = {frame_metrics["auc_roc"]}%')
print(f'2. Video Mean Probability   : Accuracy = {video_mean_metrics["accuracy"]}% | AUC = {video_mean_metrics["auc_roc"]}%')
print(f'3. Video Median Probability : Accuracy = {video_median_metrics["accuracy"]}% | AUC = {video_median_metrics["auc_roc"]}%')
print(f'4. Video Majority Hard Vote : Accuracy = {video_maj_acc}%')
print('=' * 80)

In [ ]:
# ── BENCHMARK COMPARISON TABLE & EXPORT ──────────────────────────
OUTPUT_DIR = Path('/kaggle/working/output_video_eval')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    'frame_level': frame_metrics,
    'video_mean_voting': video_mean_metrics,
    'video_median_voting': video_median_metrics,
    'video_majority_accuracy': video_maj_acc,
    'total_validation_frames': len(all_labels),
    'total_validation_videos': len(sorted_vids)
}

with open(OUTPUT_DIR / 'video_eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n' + '=' * 85)
print('FINAL COMPARISON WITH IEEE SMART-2025 PAPER (Table II)')
print('=' * 85)
rows = [
    ('Original Paper Claim (Table II: FF++)', '98.40', '97.20', '98.60', '97.60', '98.40'),
    ('EFFResNet-ViT (Frame-Level)', str(frame_metrics['accuracy']), str(frame_metrics['precision']), str(frame_metrics['recall']), str(frame_metrics['f1_score']), str(frame_metrics['auc_roc'])),
    ('EFFResNet-ViT (Video-Level Mean Voting)', str(video_mean_metrics['accuracy']), str(video_mean_metrics['precision']), str(video_mean_metrics['recall']), str(video_mean_metrics['f1_score']), str(video_mean_metrics['auc_roc'])),
    ('EFFResNet-ViT (Video-Level Median Voting)', str(video_median_metrics['accuracy']), str(video_median_metrics['precision']), str(video_median_metrics['recall']), str(video_median_metrics['f1_score']), str(video_median_metrics['auc_roc'])),
]
print(f'{"Evaluation Paradigm":<45} {"Acc%":>7} {"Prec%":>7} {"Rec%":>7} {"F1%":>7} {"AUC%":>7}')
print('-' * 85)
for r in rows:
    print(f'{r[0]:<45} {r[1]:>7} {r[2]:>7} {r[3]:>7} {r[4]:>7} {r[5]:>7}')
print('=' * 85)

# Confusion Matrix Visualization
cm_v = np.array(video_mean_metrics['confusion_matrix'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm_v, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.title(f'Video-Level Confusion Matrix (Acc: {video_mean_metrics["accuracy"]}%)')
plt.xlabel('Predicted Video Label')
plt.ylabel('Ground Truth Video Label')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'video_confusion_matrix.png', dpi=300)
plt.show()
print(f'Summary and plots saved to {OUTPUT_DIR}')